# 01 — Explore GST Entity Data

Verify S3 access and inspect the GST-registered entity data before running the indexing pipeline.

**Run this on MAESTRO SageMaker JupyterLab.**

## 1. Install dependencies

In [ ]:
# Uncomment and run if packages are not yet installed
# !pip install boto3 pandas pyarrow

## 2. Verify S3 access

In [ ]:
import boto3

S3_BUCKET = "sst-s3-gvt-agml-prodizna-d-andnwekll2vd-bucket"
S3_GST_PREFIX = "gst-registrants/"

s3 = boto3.client("s3")

# List objects under the GST prefix
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_GST_PREFIX, MaxKeys=20)

print(f"Bucket: {S3_BUCKET}")
print(f"Prefix: {S3_GST_PREFIX}")
print(f"Objects found: {response.get('KeyCount', 0)}")
print()

for obj in response.get("Contents", []):
    size_mb = obj["Size"] / (1024 * 1024)
    print(f"  {obj['Key']}  ({size_mb:.2f} MB)")

## 3. Load and inspect the data

In [ ]:
import io
import pandas as pd

# Pick the first CSV or parquet file found above.
# Update this key if your file has a different name.
DATA_KEY = response["Contents"][0]["Key"]
print(f"Loading: s3://{S3_BUCKET}/{DATA_KEY}")

body = s3.get_object(Bucket=S3_BUCKET, Key=DATA_KEY)["Body"].read()

if DATA_KEY.endswith(".parquet"):
    df = pd.read_parquet(io.BytesIO(body))
elif DATA_KEY.endswith(".csv"):
    df = pd.read_csv(io.BytesIO(body))
else:
    raise ValueError(f"Unsupported file type: {DATA_KEY}")

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

## 4. Identify the entity name column

In [ ]:
# The indexing pipeline auto-detects the column, but let's verify here.
# Update ENTITY_COLUMN if auto-detection picks the wrong one.
ENTITY_COLUMN = None  # Set to a column name string to override, e.g. "name"

candidates = ["entity_name", "name", "company_name", "business_name", "registrant_name"]
lower_cols = {c.lower(): c for c in df.columns}

if ENTITY_COLUMN:
    col = ENTITY_COLUMN
else:
    col = None
    for candidate in candidates:
        if candidate in lower_cols:
            col = lower_cols[candidate]
            break
    if col is None:
        col = df.select_dtypes(include="object").columns[0]

print(f"Using column: '{col}'")
print(f"Non-null values: {df[col].notna().sum()}")
print(f"Unique values: {df[col].nunique()}")
print()
print("Sample entity names:")
df[col].dropna().sample(min(20, len(df)), random_state=42).tolist()

## 5. Quick data quality checks

In [ ]:
names = df[col].dropna()

print(f"Total rows: {len(df)}")
print(f"Non-null names: {len(names)}")
print(f"Unique names: {names.nunique()}")
print(f"Duplicates: {len(names) - names.nunique()}")
print()
print(f"Shortest name: '{names.str.len().min()}' chars")
print(f"Longest name: '{names.str.len().max()}' chars")
print(f"Median length: {names.str.len().median():.0f} chars")

## Next step

If everything looks good, proceed to **`02_run_indexing.ipynb`** to embed all entities and build the FAISS index.